# SMPL-X body-only fit to triangulated body keypoints

Fits an SMPL-X body model (no hands/face pose -- `body_pose` only, plus
`global_orient`/`transl`/`betas`) to the triangulated MediaPipe Pose body
joints from `body_keypoint_3d.npz` (produced by
`body_keypoint_reconstruction.ipynb`).

1. Build the SMPL-X model (`v3.smplx_body_fit.build_model`).
2. Pick a handful of representative frames (`valid==True`, max `n_inliers`,
   not jitter-flagged).
3. Per frame, fit `(global_orient, transl, body_pose, betas)` via Adam against
   the 15 MediaPipe<->SMPL-X joint correspondences in
   `v3.smplx_body_fit.JOINT_CORRESPONDENCES` (direct 3D distance loss + small
   shape/pose regularizers), printing loss every 50 iterations and a final
   per-joint residual table (mm).
4. Visualize the fitted SMPL-X mesh + skeleton alongside the triangulated
   targets in viser, per representative frame.

Generated by `v3/notebooks/build_smplx_notebook.py`, which is the source of
truth -- edit that script and re-run
`python -m v3.notebooks.build_smplx_notebook`, don't edit this notebook
directly.

Runs entirely in `oak_env`.

In [ ]:
import time

import numpy as np
import torch

from v3.smplx_body_fit import build_model, get_targets, fit_frame, JOINT_CORRESPONDENCES
from v3.skeletons import SMPLX_BODY_BONES
from v3 import visualization as viz

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(DEVICE)
print(f'SMPL-X model loaded on {DEVICE}')

In [ ]:
# --- Load triangulated body keypoints, pick representative frames --------------------
d = np.load('body_keypoint_3d.npz')
joints3d_sg = d['joints3d_sg']
valid = d['valid']
n_inliers = d['n_inliers']
jitter_flag = d['jitter_flag']

# Representative frames: valid, max inlier-camera count, not jitter-flagged --
# the highest-confidence triangulations available.
candidates = np.where(valid & (n_inliers == n_inliers.max()) & ~jitter_flag)[0]
N_REP_FRAMES = 5
pick = np.linspace(0, len(candidates) - 1, N_REP_FRAMES).round().astype(int)
REP_FRAMES = candidates[pick].tolist()

print(f'{len(candidates)} candidate frames (n_inliers=={n_inliers.max()}, not jitter-flagged)')
print(f'representative frames: {REP_FRAMES}')

In [ ]:
# --- Per-frame SMPL-X fit (global_orient, transl, body_pose, betas) -------------------
N_ITERS = 300
LR = 0.05
W_BETAS = 1e-2
W_POSE = 1e-3

ALL_LABELS = [c[3] for c in JOINT_CORRESPONDENCES]

fit_results = {}
for fi in REP_FRAMES:
    print(f'=== frame {fi} (n_inliers={int(n_inliers[fi])}) ===')
    target, weights, smplx_idx, labels = get_targets(joints3d_sg[fi])
    result = fit_frame(model, target, weights, smplx_idx, DEVICE,
                        n_iters=N_ITERS, lr=LR, w_betas=W_BETAS, w_pose=W_POSE,
                        print_every=50, label=f'frame{fi}')
    result['target'] = target
    fit_results[fi] = result

    print('  per-joint residuals (mm):')
    for label, res in zip(labels, result['residuals']):
        print(f'    {label:16s}: {res * 1000:5.1f}mm')
    print(f'  mean residual: {result["residuals"].mean() * 1000:.1f}mm')

In [ ]:
# --- Viser visualization: SMPL-X mesh + skeleton vs. triangulated targets -------------
import viser

SMPLX_FACES = model.faces
MESH_COLOR = (180, 200, 230)
SKELETON_COLOR = (120, 220, 120)
TARGET_COLOR = (255, 80, 80)  # triangulated MediaPipe targets, for contrast

PELVIS_IDX = ALL_LABELS.index('pelvis')
ORIGIN_WORLD = fit_results[REP_FRAMES[0]]['target'][PELVIS_IDX]


def main():
    server = viser.ViserServer()
    running = True
    try:
        server.scene.set_up_direction('+z')
        server.gui.configure_theme(dark_mode=True)

        server.initial_camera.position = (2.5, 2.0, 1.6)
        server.initial_camera.look_at = (0.0, 0.0, 0.9)
        server.initial_camera.up = (0.0, 0.0, 1.0)

        print('Viser: open the URL shown above')

        stop_button = server.gui.add_button('Stop viewer')
        frame_dropdown = server.gui.add_dropdown(
            'Frame', options=[str(fi) for fi in REP_FRAMES], initial_value=str(REP_FRAMES[0]))

        @stop_button.on_click
        def _stop(_) -> None:
            nonlocal running
            running = False

        def render(fi):
            result = fit_results[fi]
            verts_viser = (result['vertices'] - ORIGIN_WORLD) @ viz.R_WORLD_TO_VISER.T
            server.scene.add_mesh_simple(
                '/smplx/mesh', verts_viser.astype(np.float32), SMPLX_FACES,
                color=MESH_COLOR, opacity=0.85, flat_shading=False)
            viz.add_skeleton(server, '/smplx/skeleton', result['joints'][:22], SMPLX_BODY_BONES,
                             SKELETON_COLOR, origin_world=ORIGIN_WORLD)
            targets_viser = (result['target'] - ORIGIN_WORLD) @ viz.R_WORLD_TO_VISER.T
            server.scene.add_point_cloud(
                '/targets', targets_viser.astype(np.float32),
                colors=np.full((len(targets_viser), 3), TARGET_COLOR, dtype=np.uint8),
                point_size=0.02)

        render(REP_FRAMES[0])

        @frame_dropdown.on_update
        def _on_frame_change(_) -> None:
            render(int(frame_dropdown.value))

        while running:
            time.sleep(0.05)
    except KeyboardInterrupt:
        print('Viewer interrupted.')
    finally:
        server.stop()
        print('Viser stopped.')


main()